In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

# 1. Load Environment

In [4]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

In [5]:
import transformers
import datasets
import accelerate
import peft
import bitsandbytes
import trl

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("trl:", trl.__version__)

transformers: 4.57.3
datasets: 4.0.0
accelerate: 1.12.0
peft: 0.18.0
bitsandbytes: 0.49.0
trl: 0.26.2


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. Import Libraries

In [7]:
import numpy as np
from datasets import Dataset, load_dataset, load_from_disk
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score

# 3. Load Dataset

In [8]:
name_dataset = 'AG_News'

In [9]:
dataset = load_dataset("ag_news")

In [10]:
dataset.shape

{'train': (120000, 2), 'test': (7600, 2)}

In [11]:
dataset_train = dataset['train']
pre_dataset_test = dataset['test']

In [12]:
split_test = pre_dataset_test.train_test_split(test_size=0.5, stratify_by_column = 'label', seed=42)

In [13]:
dataset_val = split_test['train']
dataset_test = split_test['test']

In [14]:
df_train = dataset_train.to_pandas()
df_val = dataset_val.to_pandas()
df_test = dataset_test.to_pandas()

**a. Analysis: Train Set**

In [15]:
df_train.shape

(120000, 2)

In [16]:
df_train['label'].value_counts()

,count
label,
2,30000
3,30000
1,30000
0,30000


In [17]:
round(df_train['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
2,25.0
3,25.0
1,25.0
0,25.0


**b. Analysis: Validation Set**

In [18]:
df_val.shape

(3800, 2)

In [19]:
df_val['label'].value_counts()

,count
label,
0,950
1,950
2,950
3,950


In [20]:
round(df_val['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
0,25.0
1,25.0
2,25.0
3,25.0


**c. Analysis: Test Set**

In [21]:
df_test.shape

(3800, 2)

In [22]:
df_test['label'].value_counts()

,count
label,
3,950
1,950
2,950
0,950


In [23]:
round(df_test['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
3,25.0
1,25.0
2,25.0
0,25.0


**d. Save dataframes**

In [24]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/01.Datasets_Creation/{name_dataset}'

In [25]:
df_train.to_csv(f'{path_save}/df_train.csv')
df_val.to_csv(f'{path_save}/df_val.csv')
df_test.to_csv(f'{path_save}/df_test.csv')

# 4. BERT

In [26]:
name_model = "bert-base-uncased"

In [27]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

In [28]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [29]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

In [30]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

In [31]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

Map:   0%|          | 0/3800 [00:00<?, ? examples/s]

In [32]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/120000 [00:00<?, ? examples/s]

In [33]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/3800 [00:00<?, ? examples/s]

In [34]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/3800 [00:00<?, ? examples/s]

# 5. DistilBERT

In [35]:
name_model = "distilbert-base-uncased"

In [36]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

In [37]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [38]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

In [39]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

In [40]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

Map:   0%|          | 0/3800 [00:00<?, ? examples/s]

In [41]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/120000 [00:00<?, ? examples/s]

In [42]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/3800 [00:00<?, ? examples/s]

In [43]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/3800 [00:00<?, ? examples/s]

# 6. RoBERTa

In [44]:
name_model = "roberta-base"

In [45]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

In [46]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [47]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

In [48]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

In [49]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

Map:   0%|          | 0/3800 [00:00<?, ? examples/s]

In [50]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/120000 [00:00<?, ? examples/s]

In [51]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/3800 [00:00<?, ? examples/s]

In [52]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/3800 [00:00<?, ? examples/s]

# 7. Execution Time

In [53]:
end_notebook = time.time()

In [54]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 0m 35.50s
